# SO-101 エサやり／片付け — SmolVLAで「言葉で動作を切り替える」（Colab用）

シーンは常に同じ（エサ入りカップ・青い皿・緑のマット）。
- **feed**（"feed the pet" など）: カップを皿の上で傾けて注ぐ
- **put**（"put the cup away" など）: カップを緑のマットに立てて置く

カップをつかんで持ち上げるまでは同じ動きなので、どちらへ進むかは**言葉だけ**で決まります。

ランタイム → ランタイムのタイプを変更 → **GPU（T4以上）**。事前に `feed_smolvla_lang.zip` を `マイドライブ/physical_ai/` に置いてください。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/work && cd /content/work && unzip -q -o /content/drive/MyDrive/physical_ai/feed_smolvla_lang.zip
%cd /content/work/feed_smolvla_lang
!ls

In [ ]:
# 依存パッケージ（数分）
!pip install -q "lerobot[smolvla,dataset,training]" mujoco imageio imageio-ffmpeg
!python -c "import av, accelerate, lerobot; print('lerobot', lerobot.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# MuJoCoをGPU(EGL)で描画
import os
os.environ['MUJOCO_GL'] = 'egl'
%env MUJOCO_GL=egl
!python -c "from feed_env import FeedEnv; e=FeedEnv(); o=e.reset(0); print('sim ok', o['top'].shape)"
!python -c "import json; i=json.load(open('data/so101_feed_put/meta/info.json')); print('episodes', i['total_episodes'], 'frames', i['total_frames'])"

## 1. 学習
2タスクなので1回目（10,000）より長めの15,000ステップ。T4で3〜4時間が目安。5,000ステップごとにチェックポイントを保存します。

Colabの接続が切れたときに備えて、学習後すぐにドライブへコピーするセルも続けて実行してください。

In [ ]:
!STEPS=15000 BATCH=16 bash train_smolvla.sh

In [ ]:
# 学習済みモデルをすぐドライブに退避
!mkdir -p /content/drive/MyDrive/physical_ai/smolvla_lang_results
!cp -r outputs/smolvla_feed_put/checkpoints/last/pretrained_model /content/drive/MyDrive/physical_ai/smolvla_lang_results/

## 2. 評価：同じシーンに対して指示文だけ変える
feed / put それぞれについて、**学習時の言い回し**と**学習に使っていない言い回し**で20シードずつ（合計80エピソード、40〜50分）。

- `success`: 指示どおりのタスクを成功
- `取り違え`: 指示と違う動作をした（feedなのにマットに置いた／putなのに皿に注いだ）

In [ ]:
CK='outputs/smolvla_feed_put/checkpoints/last/pretrained_model'
!python eval_smolvla_lang.py --ckpt $CK --n 20

## 3. 同じシーンで2つの指示を比べる動画

In [ ]:
!python eval_smolvla_lang.py --ckpt $CK --video_seed 50003
from IPython.display import Video, display
for t in ('feed', 'put'):
    display(Video(f'eval_results/seed50003_{t}.mp4', embed=True, width=480))

## 4. 好きな言い回しで試す
`--task` には正解のタスク（feed / put）を指定します。

In [ ]:
!python eval_smolvla_lang.py --ckpt $CK --n 5 --task put --phrase "could you put that cup back on the mat"
!python eval_smolvla_lang.py --ckpt $CK --n 5 --task feed --phrase "the cat is hungry"

## 5. 結果をドライブに保存

In [ ]:
!cp -r eval_results /content/drive/MyDrive/physical_ai/smolvla_lang_results/
!ls /content/drive/MyDrive/physical_ai/smolvla_lang_results/eval_results